# 06. legacy LightGBM 위험도 및 리드타임 모델

이 노트북은 05번 Isolation Forest anomaly score와 04번 feature table을 결합해 기존 LightGBM 기반 위험도 모델을 재현한다.

현재 역할:
- 이 노트북은 canonical ML 구현이 아니라 legacy comparison baseline이다.
- paper-aligned Autoencoder 방향으로 전환하기 전 기존 결과를 보존한다.
- Agent / Priority Engine 계약과 충돌했던 지점을 비교용으로 남긴다.

기존 역할 구분:
- Isolation Forest: 정상 운전 패턴과 다른 이상징후를 찾고 `anomaly_score`를 산출한다.
- LightGBM: 이상징후와 센서 feature가 고장신고 전 위험구간 패턴과 유사한지 판단해 `risk_probability`를 산출한다.

주의:
- `risk_probability`는 고장 확정값이 아니다.
- `faults.csv` 기준 라벨은 실제 고장 발생 시점이 아니라 고장신고 전 위험구간 유사도 학습용 라벨이다.


## 0-A. legacy 보강 실험 원칙

현재 이 노트북은 legacy LightGBM 체인의 보강 기록을 보존한다.

1. 03번 upstream normal 기준 재검토
2. holdout drift 상위 temperature 계열 feature 제외 실험
3. paper-aligned canonical 전환 전 비교 baseline 보존

이 노트북에서는 기본 guarded baseline을 유지한 채, drift feature 제외 실험을 토글로 재현할 수 있게 둔다.
기본값은 비활성화이며, 실험을 켰을 때만 feature 집합과 metadata가 바뀐다.


In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, precision_recall_fscore_support, roc_auc_score

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('pyproject.toml not found. Run from the project tree.')


PROJECT_ROOT = find_project_root(Path.cwd())
FEATURE_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_features'
BASELINE_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_baseline'
ALIGNMENT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'label_alignment'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_risk'
MODEL_DIR = OUTPUT_DIR / 'models'

TRAINABLE_WINDOWS_PATH = FEATURE_DIR / 'trainable_windows.csv'
ML_WINDOW_DATASET_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ml_windows' / 'ml_window_dataset.csv'
FEATURE_COLUMNS_PATH = FEATURE_DIR / 'feature_columns.csv'
ANOMALY_SCORES_PATH = BASELINE_DIR / 'anomaly_baseline_scores.csv'
FAULT_ALIGNMENT_PATH = ALIGNMENT_DIR / 'fault_alignment.csv'
DISTURBANCE_ALIGNMENT_PATH = ALIGNMENT_DIR / 'disturbance_alignment.csv'

RISK_SCORES_PATH = OUTPUT_DIR / 'lgbm_risk_scores.csv'
RISK_METRICS_PATH = OUTPUT_DIR / 'lgbm_risk_metrics.csv'
RISK_THRESHOLDS_PATH = OUTPUT_DIR / 'lgbm_risk_thresholds.csv'
THRESHOLD_SELECTION_PATH = OUTPUT_DIR / 'lgbm_threshold_selection.csv'
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / 'lgbm_feature_importance.csv'
LEAKAGE_AUDIT_PATH = OUTPUT_DIR / 'event_split_leakage_audit.csv'
RUN_CONSISTENCY_PATH = OUTPUT_DIR / 'lgbm_run_consistency.csv'
MODEL_PATH = MODEL_DIR / 'lightgbm_risk_model.joblib'
MODEL_METADATA_PATH = MODEL_DIR / 'risk_model_metadata.json'

PRIMARY_SPLIT_COLUMN = 'split_event_regime_based'
EVALUATION_SPLIT_COLUMNS = ['split_event_regime_based', 'split_event_based', 'split_regime_based', 'split_time_based', 'split_substation_based']
SPLIT_VALUES = ['train', 'validation', 'holdout']
RANDOM_STATE = 42
MODEL_VERSION = 'lgbm_risk_06_event_days_v3'
ENABLE_DRIFT_FEATURE_EXCLUSION_EXPERIMENT = False
DRIFT_FEATURE_CANDIDATES = [
    'p_net_return_temperature__mean',
    'p_net_return_temperature__last',
    'p_net_return_temperature__first',
    'p_net_return_temperature__min',
    's_hc1_supply_temperature__std',
]
EVENT_CONTEXT_SENTINEL_DAYS = 9999.0
RECENT_EVENT_WINDOWS_DAYS = [30, 60, 90]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'feature input: {TRAINABLE_WINDOWS_PATH}')
print(f'anomaly score input: {ANOMALY_SCORES_PATH}')
print(f'fault alignment input: {FAULT_ALIGNMENT_PATH}')
print(f'disturbance alignment input: {DISTURBANCE_ALIGNMENT_PATH}')
print(f'output directory: {OUTPUT_DIR}')
print(f'drift feature exclusion experiment: {ENABLE_DRIFT_FEATURE_EXCLUSION_EXPERIMENT}')




feature input: C:\Project3\HeatGrid_Agent\data\processed\ml_features\trainable_windows.csv
anomaly score input: C:\Project3\HeatGrid_Agent\data\processed\ml_baseline\anomaly_baseline_scores.csv
fault alignment input: C:\Project3\HeatGrid_Agent\data\processed\label_alignment\fault_alignment.csv
disturbance alignment input: C:\Project3\HeatGrid_Agent\data\processed\label_alignment\disturbance_alignment.csv
output directory: C:\Project3\HeatGrid_Agent\data\processed\ml_risk
drift feature exclusion experiment: False


## 1. 입력 데이터 로딩 및 결합

04번의 학습 feature와 05번의 Isolation Forest anomaly score를 같은 window key로 결합한다.


In [2]:
trainable_windows = pd.read_csv(TRAINABLE_WINDOWS_PATH)
ml_window_dataset = pd.read_csv(ML_WINDOW_DATASET_PATH)
feature_columns = pd.read_csv(FEATURE_COLUMNS_PATH)
anomaly_scores = pd.read_csv(ANOMALY_SCORES_PATH)
fault_alignment = pd.read_csv(FAULT_ALIGNMENT_PATH)
disturbance_alignment = pd.read_csv(DISTURBANCE_ALIGNMENT_PATH)

KEY_COLUMNS = ['manufacturer', 'substation_id', 'window_start', 'window_end']
TRAINING_CONTROL_COLUMNS = [
    column for column in [
        'use_for_supervised_training',
        'normal_reference_outlier',
        'normal_reference_outlier_count',
        'normal_reference_filter_reason',
    ]
    if column in ml_window_dataset.columns
]
if TRAINING_CONTROL_COLUMNS:
    trainable_windows = trainable_windows.drop(columns=[column for column in TRAINING_CONTROL_COLUMNS if column in trainable_windows.columns])
    trainable_windows = trainable_windows.merge(
        ml_window_dataset[KEY_COLUMNS + TRAINING_CONTROL_COLUMNS],
        on=KEY_COLUMNS,
        how='left',
        validate='one_to_one',
    )

ANOMALY_COLUMNS = KEY_COLUMNS + ['anomaly_score', 'anomaly_threshold', 'anomaly_label', 'iforest_anomaly_score']
available_anomaly_columns = [column for column in ANOMALY_COLUMNS if column in anomaly_scores.columns]

modeling_df = trainable_windows.merge(
    anomaly_scores[available_anomaly_columns],
    on=KEY_COLUMNS,
    how='left',
    validate='one_to_one',
)

if modeling_df['anomaly_score'].isna().any():
    missing_count = int(modeling_df['anomaly_score'].isna().sum())
    raise ValueError(f'anomaly_score missing after merge: {missing_count}')

if 'use_for_supervised_training' in modeling_df.columns:
    supervised_filter_mask = modeling_df['use_for_supervised_training'].fillna(True).astype(bool)
    removed_supervised_rows = int((~supervised_filter_mask).sum())
    modeling_df = modeling_df.loc[supervised_filter_mask].copy()
    print(f'supervised filter removed rows across all splits: {removed_supervised_rows}')

modeling_df['risk_target'] = (modeling_df['label'] == 'pre_fault').astype(int)
for column in ['window_end', 'window_start']:
    modeling_df[column] = pd.to_datetime(modeling_df[column], errors='coerce')

for column in ['report_date', 'window_start', 'window_end']:
    if column in fault_alignment.columns:
        fault_alignment[column] = pd.to_datetime(fault_alignment[column], errors='coerce')
for column in ['event_start', 'window_start', 'window_end']:
    if column in disturbance_alignment.columns:
        disturbance_alignment[column] = pd.to_datetime(disturbance_alignment[column], errors='coerce')


def previous_event_features(windows: pd.DataFrame, events: pd.DataFrame, event_time_column: str, prefix: str) -> pd.DataFrame:
    result = pd.DataFrame(index=windows.index)
    days_column = f'days_since_last_{prefix}_event'
    has_column = f'has_previous_{prefix}_event'
    result[days_column] = EVENT_CONTEXT_SENTINEL_DAYS
    result[has_column] = 0
    for days in RECENT_EVENT_WINDOWS_DAYS:
        result[f'recent_{prefix}_{days}d'] = 0

    required_columns = {'manufacturer', 'substation_id', event_time_column}
    if events.empty or not required_columns.issubset(events.columns):
        return result

    clean_events = events.dropna(subset=[event_time_column]).copy()
    if clean_events.empty:
        return result

    event_groups = {
        key: group[event_time_column].sort_values().to_numpy(dtype='datetime64[ns]')
        for key, group in clean_events.groupby(['manufacturer', 'substation_id'], dropna=False)
    }

    for key, group_index in windows.groupby(['manufacturer', 'substation_id'], dropna=False).groups.items():
        event_times = event_groups.get(key)
        if event_times is None or len(event_times) == 0:
            continue
        window_times = windows.loc[group_index, 'window_start'].to_numpy(dtype='datetime64[ns]')
        positions = np.searchsorted(event_times, window_times, side='left') - 1
        valid = positions >= 0
        if not valid.any():
            continue
        target_index = np.asarray(list(group_index))[valid]
        deltas = (window_times[valid] - event_times[positions[valid]]) / np.timedelta64(1, 'D')
        deltas = np.maximum(deltas.astype(float), 0.0)
        result.loc[target_index, days_column] = deltas
        result.loc[target_index, has_column] = 1
        for days in RECENT_EVENT_WINDOWS_DAYS:
            result.loc[target_index, f'recent_{prefix}_{days}d'] = (deltas <= days).astype('int8')
    return result

usable_fault_events = fault_alignment.copy()
if 'is_usable' in usable_fault_events.columns:
    usable_fault_events = usable_fault_events[usable_fault_events['is_usable'].eq(True)].copy()
usable_fault_events = usable_fault_events.rename(columns={'report_date': 'event_time'})
usable_fault_events = usable_fault_events[['manufacturer', 'substation_id', 'event_time']].dropna(subset=['event_time'])

usable_disturbances = disturbance_alignment.copy()
if 'is_usable' in usable_disturbances.columns:
    usable_disturbances = usable_disturbances[usable_disturbances['is_usable'].eq(True)].copy()
usable_disturbances = usable_disturbances.rename(columns={'event_start': 'event_time'})
usable_disturbances = usable_disturbances[['manufacturer', 'substation_id', 'disturbance_type', 'event_time']].dropna(subset=['event_time'])
usable_task_events = usable_disturbances[~usable_disturbances['disturbance_type'].fillna('').str.lower().eq('fault')].copy()
usable_any_events = pd.concat(
    [
        usable_fault_events.assign(event_source='fault'),
        usable_task_events[['manufacturer', 'substation_id', 'event_time']].assign(event_source='task'),
    ],
    ignore_index=True,
)

event_context_frames = [
    previous_event_features(modeling_df, usable_fault_events, 'event_time', 'fault'),
    previous_event_features(modeling_df, usable_task_events, 'event_time', 'task'),
    previous_event_features(modeling_df, usable_any_events, 'event_time', 'any'),
]
for event_context in event_context_frames:
    for column in event_context.columns:
        modeling_df[column] = event_context[column]

modeling_df['post_fault_recent_90d'] = modeling_df['recent_fault_90d'].astype('int8')
modeling_df['post_task_recent_90d'] = modeling_df['recent_task_90d'].astype('int8')

fault_event_order = (
    modeling_df.dropna(subset=['fault_event_id'])
    .groupby('fault_event_id')['window_end']
    .max()
    .sort_values()
)
fault_event_ids = fault_event_order.index.to_list()
train_event_end = int(np.floor(len(fault_event_ids) * 0.70))
validation_event_end = int(np.floor(len(fault_event_ids) * 0.85))
fault_event_split_map = {}
for event_index, fault_event_id in enumerate(fault_event_ids):
    if event_index < train_event_end:
        fault_event_split_map[fault_event_id] = 'train'
    elif event_index < validation_event_end:
        fault_event_split_map[fault_event_id] = 'validation'
    else:
        fault_event_split_map[fault_event_id] = 'holdout'

modeling_df['split_event_based'] = modeling_df['split_time_based']
base_regime_split = modeling_df['split_regime_based'] if 'split_regime_based' in modeling_df.columns else modeling_df['split_time_based']
modeling_df['split_event_regime_based'] = base_regime_split
fault_event_mask = modeling_df['fault_event_id'].notna()
modeling_df.loc[fault_event_mask, 'split_event_based'] = modeling_df.loc[fault_event_mask, 'fault_event_id'].map(fault_event_split_map)
modeling_df.loc[fault_event_mask, 'split_event_regime_based'] = modeling_df.loc[fault_event_mask, 'fault_event_id'].map(fault_event_split_map)

for split_column in ['split_event_based', 'split_event_regime_based']:
    if modeling_df[split_column].isna().any():
        missing_count = int(modeling_df[split_column].isna().sum())
        raise ValueError(f'{split_column} missing rows: {missing_count}')

display(
    modeling_df[
        [
            'label',
            'risk_target',
            'anomaly_score',
            'maintenance_related',
            'disturbance_count',
            'days_since_last_fault_event',
            'days_since_last_task_event',
            'recent_fault_90d',
            'configuration_type',
            'split_event_based',
            'split_event_regime_based',
        ]
    ].head()
)
display(pd.crosstab(modeling_df[PRIMARY_SPLIT_COLUMN], modeling_df['label']))



supervised filter removed rows across all splits: 164


,label,risk_target,anomaly_score,maintenance_related,disturbance_count,days_since_last_fault_event,days_since_last_task_event,recent_fault_90d,configuration_type,split_event_based,split_event_regime_based
0,normal,0,0.416895,False,0,9999.0,9999.0,0,SH + DHW,validation,train
1,normal,0,0.430012,False,0,9999.0,9999.0,0,SH + DHW,validation,train
2,normal,0,0.426032,False,0,9999.0,9999.0,0,SH + DHW,validation,train
3,normal,0,0.440612,False,0,9999.0,9999.0,0,SH + DHW,validation,validation
4,normal,0,0.424733,False,0,9999.0,9999.0,0,SH + DHW,validation,validation


label,normal,pre_fault
split_event_regime_based,,
holdout,214,86
train,1224,407
validation,288,143


## 2. LightGBM feature 구성

LightGBM 입력은 04번에서 선택한 sensor feature에 05번 anomaly score와 정비/작업 이력 feature를 추가한다.
라벨, fault id, lead time, normal event 여부처럼 직접적인 정답 또는 출처 단서는 feature에서 제외한다.


In [3]:

selected_feature_columns = (
    feature_columns.loc[feature_columns['selected_for_baseline'] == True, 'column_name']
    .dropna()
    .tolist()
)

CUMULATIVE_ABSOLUTE_FEATURES = {
    f'p_net_meter_{meter}__{stat}'
    for meter in ['energy', 'volume']
    for stat in ['first', 'last', 'min', 'max', 'mean']
}
excluded_feature_columns = [column for column in selected_feature_columns if column in CUMULATIVE_ABSOLUTE_FEATURES]
guarded_feature_columns = [column for column in selected_feature_columns if column not in CUMULATIVE_ABSOLUTE_FEATURES]
drift_excluded_feature_columns = []
if ENABLE_DRIFT_FEATURE_EXCLUSION_EXPERIMENT:
    drift_excluded_feature_columns = [
        column for column in guarded_feature_columns
        if column in DRIFT_FEATURE_CANDIDATES
    ]
    guarded_feature_columns = [
        column for column in guarded_feature_columns
        if column not in drift_excluded_feature_columns
    ]
    excluded_feature_columns = [*excluded_feature_columns, *drift_excluded_feature_columns]

has_manufacturer_onehot = any(column.startswith('manufacturer__is__') for column in selected_feature_columns)
has_configuration_onehot = any(column.startswith('configuration_type__is__') for column in selected_feature_columns)
if 'manufacturer' in modeling_df.columns and not has_manufacturer_onehot:
    modeling_df['manufacturer_code'] = modeling_df['manufacturer'].astype('category').cat.codes.astype('int16')
if 'configuration_type' in modeling_df.columns and not has_configuration_onehot:
    modeling_df['configuration_code'] = (
        modeling_df['configuration_type']
        .fillna('missing')
        .astype('category')
        .cat.codes
        .astype('int16')
    )

event_context_model_feature_candidates = [
    'days_since_last_fault_event',
    'days_since_last_task_event',
    'days_since_last_any_event',
]
context_feature_candidates = [
    'anomaly_score',
    'disturbance_count',
    'maintenance_related',
    *event_context_model_feature_candidates,
]
group_feature_candidates = []
if not has_manufacturer_onehot and 'manufacturer_code' in modeling_df.columns:
    group_feature_candidates.append('manufacturer_code')
if not has_configuration_onehot and 'configuration_code' in modeling_df.columns:
    group_feature_candidates.append('configuration_code')

context_feature_columns = [column for column in context_feature_candidates if column in modeling_df.columns]
group_feature_columns = [column for column in group_feature_candidates if column in modeling_df.columns]

model_feature_columns = []
for column in [*guarded_feature_columns, *context_feature_columns, *group_feature_columns]:
    if column not in model_feature_columns:
        model_feature_columns.append(column)

X_all = modeling_df[model_feature_columns].copy()
for column in X_all.columns:
    if X_all[column].dtype == 'bool':
        X_all[column] = X_all[column].astype('int8')
    elif X_all[column].dtype == 'object':
        X_all[column] = pd.to_numeric(X_all[column], errors='coerce')

if X_all.isna().any().any():
    missing_summary = X_all.isna().sum()
    missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
    raise ValueError('LightGBM input features contain missing values:\n' + str(missing_summary.head(20)))

y_all = modeling_df['risk_target'].astype(int)
train_mask = modeling_df[PRIMARY_SPLIT_COLUMN].eq('train')
validation_mask = modeling_df[PRIMARY_SPLIT_COLUMN].eq('validation')

X_train = X_all.loc[train_mask]
y_train = y_all.loc[train_mask]
X_validation = X_all.loc[validation_mask]
y_validation = y_all.loc[validation_mask]

print(f'original selected feature count: {len(selected_feature_columns)}')
print(f'excluded cumulative absolute feature count: {len(excluded_feature_columns)}')
print(f'drift excluded feature count: {len(drift_excluded_feature_columns)}')
print(f'context feature count: {len(context_feature_columns)}')
print(f'group feature count: {len(group_feature_columns)}')
print(f'event context model feature count: {len([c for c in event_context_model_feature_candidates if c in context_feature_columns])}')
print(f'final feature count: {len(model_feature_columns)}')
print(f'train rows: {len(X_train)}')
print(f'train positive ratio: {y_train.mean():.4f}')
print(f'validation positive ratio: {y_validation.mean():.4f}')
print('excluded features:', excluded_feature_columns)
print('drift excluded features:', drift_excluded_feature_columns)
print('event context model features:', [c for c in event_context_model_feature_candidates if c in context_feature_columns])


original selected feature count: 195
excluded cumulative absolute feature count: 10
drift excluded feature count: 0
context feature count: 6
group feature count: 0
event context model feature count: 3
final feature count: 189
train rows: 1631
train positive ratio: 0.2495
validation positive ratio: 0.3318
excluded features: ['p_net_meter_energy__first', 'p_net_meter_energy__last', 'p_net_meter_energy__max', 'p_net_meter_energy__mean', 'p_net_meter_energy__min', 'p_net_meter_volume__first', 'p_net_meter_volume__last', 'p_net_meter_volume__max', 'p_net_meter_volume__mean', 'p_net_meter_volume__min']
drift excluded features: []
event context model features: ['days_since_last_fault_event', 'days_since_last_task_event', 'days_since_last_any_event']


## 3. LightGBM 학습

학습은 `split_event_based == train` 기준으로 수행한다.
pre_fault 행은 `fault_event_id` 단위로 split해 같은 fault event가 학습/평가에 동시에 들어가지 않게 한다.
클래스 불균형은 `class_weight='balanced'`로 1차 보정한다.


In [4]:
risk_model = LGBMClassifier(
    objective='binary',
    n_estimators=150,
    learning_rate=0.04,
    num_leaves=15,
    min_child_samples=50,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

risk_model.fit(
    X_train,
    y_train,
    eval_set=[(X_validation, y_validation)],
    eval_metric='average_precision',
)

risk_probability = risk_model.predict_proba(X_all)[:, 1]
modeling_df['risk_probability'] = risk_probability
modeling_df['risk_score'] = risk_probability

display(modeling_df[['label', 'risk_probability', 'anomaly_score']].head())


,label,risk_probability,anomaly_score
0,normal,0.156220,0.416895
1,normal,0.090324,0.430012
2,normal,0.108072,0.426032
3,normal,0.413477,0.440612
4,normal,0.119385,0.424733


## 4. 위험도 threshold와 risk level

운영용 `high` threshold는 event validation split에서 F1이 가장 높은 확률 기준으로 선택한다.
`medium`은 high threshold의 절반, `critical`은 high threshold와 0.90 중 큰 값으로 둔다.
이 값은 우선 점검 후보를 나누기 위한 참고값이며 고장 확정 기준이 아니다.


In [5]:
def threshold_metrics_at(frame: pd.DataFrame, threshold: float) -> dict:
    y_true = frame['risk_target'].to_numpy()
    y_pred = (frame['risk_probability'].to_numpy() >= threshold).astype(int)
    negative_mask = y_true == 0
    if negative_mask.sum() == 0:
        fpr = np.nan
    else:
        fpr = float(((y_pred == 1) & negative_mask).sum() / negative_mask.sum())
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average='binary',
        zero_division=0,
    )
    return {
        'threshold': float(threshold),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'false_positive_rate': fpr,
    }


validation_for_threshold = modeling_df.loc[validation_mask].copy()
threshold_candidates = np.array([0.22, 0.40, 0.42, 0.44, 0.45, 0.90])
threshold_selection_df = pd.DataFrame(
    [threshold_metrics_at(validation_for_threshold, threshold) for threshold in threshold_candidates]
)
threshold_selection_df = threshold_selection_df.sort_values(
    ['f1', 'recall', 'precision'],
    ascending=[False, False, False],
).reset_index(drop=True)

selected_high_threshold = 0.44
risk_thresholds = {
    'medium': 0.22,
    'high': selected_high_threshold,
    'critical': 0.90,
}


def assign_risk_level(score: float) -> str:
    if score >= risk_thresholds['critical']:
        return 'critical'
    if score >= risk_thresholds['high']:
        return 'high'
    if score >= risk_thresholds['medium']:
        return 'medium'
    return 'low'


modeling_df['risk_level'] = modeling_df['risk_probability'].map(assign_risk_level)

thresholds_df = pd.DataFrame(
    [
        {'risk_level': level, 'threshold': threshold, 'basis': 'follow-up fixed operating threshold' if level == 'high' else 'fixed operating threshold set'}
        for level, threshold in risk_thresholds.items()
    ]
)

print('선택된 high threshold:', selected_high_threshold)
display(threshold_selection_df.head(10))
display(thresholds_df)
display(pd.crosstab(modeling_df[PRIMARY_SPLIT_COLUMN], modeling_df['risk_level']))


선택된 high threshold: 0.44


,threshold,precision,recall,f1,false_positive_rate
0,0.42,0.731034,0.741259,0.736111,0.135417
1,0.40,0.726027,0.741259,0.733564,0.138889
2,0.45,0.746377,0.720280,0.733096,0.121528
3,0.44,0.737589,0.727273,0.732394,0.128472
4,0.22,0.532338,0.748252,0.622093,0.326389
5,0.90,1.000000,0.321678,0.486772,0.000000


,risk_level,threshold,basis
0,medium,0.22,fixed operating threshold set
1,high,0.44,follow-up fixed operating threshold
2,critical,0.90,fixed operating threshold set


risk_level,critical,high,low,medium
split_event_regime_based,,,,
holdout,21,63,141,75
train,403,4,1222,2
validation,46,95,230,60


## 5. 평가

기본 평가는 time split과 substation split을 모두 확인한다.
lead time bucket별 성능도 별도로 본다.


In [6]:
def safe_roc_auc(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_score))


def safe_average_precision(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(average_precision_score(y_true, y_score))


def false_positive_rate(y_true, y_pred):
    negative_mask = y_true == 0
    if negative_mask.sum() == 0:
        return np.nan
    return float(((y_pred == 1) & negative_mask).sum() / negative_mask.sum())


def binary_metrics(frame: pd.DataFrame, group_name: str, group_value: str) -> dict:
    y_true = frame['risk_target'].to_numpy()
    y_score = frame['risk_probability'].to_numpy()
    y_pred = (frame['risk_level'].isin(['high', 'critical'])).astype(int).to_numpy()
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average='binary',
        zero_division=0,
    )
    return {
        'group_name': group_name,
        'group_value': group_value,
        'row_count': len(frame),
        'normal_count': int((frame['label'] == 'normal').sum()),
        'pre_fault_count': int((frame['label'] == 'pre_fault').sum()),
        'roc_auc': safe_roc_auc(y_true, y_score),
        'average_precision': safe_average_precision(y_true, y_score),
        'precision_high_or_critical': float(precision),
        'recall_high_or_critical': float(recall),
        'f1_high_or_critical': float(f1),
        'false_positive_rate_high_or_critical': false_positive_rate(y_true, y_pred),
    }


metric_rows = []
for split_column in EVALUATION_SPLIT_COLUMNS:
    for split_value in SPLIT_VALUES:
        split_df = modeling_df.loc[modeling_df[split_column] == split_value].copy()
        metric_rows.append(binary_metrics(split_df, split_column, split_value))

lead_time_bins = [-0.001, 6, 24, 72, 168]
lead_time_labels = ['0-6h', '6-24h', '1-3d', '3-7d']
modeling_df['lead_time_bucket'] = pd.cut(
    modeling_df['estimated_lead_time_hours'],
    bins=lead_time_bins,
    labels=lead_time_labels,
)

for bucket in lead_time_labels:
    bucket_df = modeling_df.loc[(modeling_df['lead_time_bucket'] == bucket) | (modeling_df['label'] == 'normal')].copy()
    metric_rows.append(binary_metrics(bucket_df, 'lead_time_bucket_vs_normal', bucket))

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)


,group_name,group_value,row_count,normal_count,pre_fault_count,roc_auc,average_precision,precision_high_or_critical,recall_high_or_critical,f1_high_or_critical,false_positive_rate_high_or_critical
0,split_event_regime_based,train,1631,1224,407,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
1,split_event_regime_based,validation,431,288,143,0.778555,0.791936,0.737589,0.727273,0.732394,0.128472
2,split_event_regime_based,holdout,300,214,86,0.762769,0.619653,0.547619,0.534884,0.541176,0.177570
3,split_event_based,train,1642,1235,407,1.000000,1.000000,0.966746,1.000000,0.983092,0.011336
4,split_event_based,validation,383,240,143,0.799184,0.839694,0.904348,0.727273,0.806202,0.045833
5,split_event_based,holdout,337,251,86,0.772352,0.581429,0.479167,0.534884,0.505495,0.199203
6,split_regime_based,train,1668,1224,444,0.990619,0.984523,1.000000,0.876126,0.933974,0.000000
7,split_regime_based,validation,382,288,94,0.970301,0.938675,0.699187,0.914894,0.792627,0.128472
8,split_regime_based,holdout,312,214,98,0.860528,0.866928,0.683333,0.836735,0.752294,0.177570
9,split_time_based,train,1652,1235,417,0.996326,0.991834,0.966019,0.954436,0.960193,0.011336


## 6. 근거 feature와 관련 이력 생성

`main_abnormal_features`는 train normal 평균 대비 z-score가 큰 feature를 사용한다.
`model_explanation_features`는 LightGBM 전역 feature importance 상위 feature를 사용한다.


In [7]:
feature_importance_df = pd.DataFrame(
    {
        'feature': model_feature_columns,
        'importance': risk_model.feature_importances_,
    }
).sort_values('importance', ascending=False).reset_index(drop=True)
feature_importance_df['rank'] = np.arange(1, len(feature_importance_df) + 1)

global_explanation_features = feature_importance_df.head(10)['feature'].tolist()
global_explanation_text = '|'.join(global_explanation_features)

train_normal_mask = train_mask & modeling_df['label'].eq('normal')
normal_mean = X_all.loc[train_normal_mask].mean()
normal_std = X_all.loc[train_normal_mask].std().replace(0, np.nan).fillna(1.0)
z_scores = ((X_all - normal_mean) / normal_std).abs()


def top_abnormal_features(row) -> str:
    top = row.sort_values(ascending=False).head(5)
    return '|'.join(top.index.tolist())


modeling_df['main_abnormal_features'] = z_scores.apply(top_abnormal_features, axis=1)
modeling_df['model_explanation_features'] = global_explanation_text


def related_fault_history(row) -> str:
    payload = {
        'fault_event_id': None if pd.isna(row.get('fault_event_id')) else int(row.get('fault_event_id')),
        'fault_label': None if pd.isna(row.get('fault_label')) else str(row.get('fault_label')),
        'estimated_lead_time_hours': None if pd.isna(row.get('estimated_lead_time_hours')) else float(row.get('estimated_lead_time_hours')),
        'days_since_last_fault_event': float(row.get('days_since_last_fault_event', EVENT_CONTEXT_SENTINEL_DAYS)),
        'recent_fault_90d': int(row.get('recent_fault_90d', 0)),
    }
    if row['label'] == 'pre_fault' or payload['recent_fault_90d']:
        return json.dumps([payload], ensure_ascii=False)
    return '[]'


def related_disturbance_history(row) -> str:
    payload = {
        'maintenance_related': bool(row.get('maintenance_related')),
        'disturbance_count': int(row.get('disturbance_count', 0)),
        'days_since_last_task_event': float(row.get('days_since_last_task_event', EVENT_CONTEXT_SENTINEL_DAYS)),
        'recent_task_90d': int(row.get('recent_task_90d', 0)),
        'days_since_last_any_event': float(row.get('days_since_last_any_event', EVENT_CONTEXT_SENTINEL_DAYS)),
        'recent_any_90d': int(row.get('recent_any_90d', 0)),
    }
    if payload['maintenance_related'] or payload['disturbance_count'] > 0 or payload['recent_task_90d'] or payload['recent_any_90d']:
        return json.dumps([payload], ensure_ascii=False)
    return '[]'


modeling_df['related_fault_history'] = modeling_df.apply(related_fault_history, axis=1)
modeling_df['related_disturbance_history'] = modeling_df.apply(related_disturbance_history, axis=1)

display(feature_importance_df.head(20))
display(modeling_df[['risk_probability', 'risk_level', 'main_abnormal_features', 'model_explanation_features']].head())



,feature,importance,rank
0,days_since_last_any_event,172,1
1,days_since_last_task_event,127,2
2,doy_sin,126,3
3,day_of_year,123,4
4,doy_cos,115,5
5,days_since_last_fault_event,86,6
6,anomaly_score,74,7
7,s_dhw_upper_storage_temperature__max,63,8
8,p_net_supply_temperature__mean,56,9
9,network_temperature_gap__mean,56,10


,risk_probability,risk_level,main_abnormal_features,model_explanation_features
0,0.156220,low,s_dhw_lower_storage_temperature__max|dhw_suppl...,days_since_last_any_event|days_since_last_task...
1,0.090324,low,s_dhw_supply_temperature_setpoint__delta|dhw_s...,days_since_last_any_event|days_since_last_task...
2,0.108072,low,p_net_return_temperature__std|s_hc1_supply_tem...,days_since_last_any_event|days_since_last_task...
3,0.413477,medium,s_dhw_supply_temperature_setpoint__delta|p_net...,days_since_last_any_event|days_since_last_task...
4,0.119385,low,days_since_last_task_event|days_since_last_any...,days_since_last_any_event|days_since_last_task...


## 7. event split leakage 점검

같은 fault event가 여러 split에 나뉘면 event 단위 성능 해석이 왜곡될 수 있다.
06번에서는 `split_event_based`를 추가해 fault event 단위 leakage를 제거하고, 기존 split들과 비교 audit table을 저장한다.


In [8]:
leakage_rows = []
fault_rows = modeling_df.dropna(subset=['fault_event_id']).copy()
for split_column in EVALUATION_SPLIT_COLUMNS:
    grouped = fault_rows.groupby('fault_event_id')[split_column].agg(lambda values: sorted(set(values)))
    for fault_event_id, split_values in grouped.items():
        leakage_rows.append(
            {
                'split_column': split_column,
                'fault_event_id': int(fault_event_id),
                'split_values': '|'.join(split_values),
                'split_count': len(split_values),
                'cross_split': len(split_values) > 1,
            }
        )

leakage_audit_df = pd.DataFrame(leakage_rows)
display(leakage_audit_df.groupby('split_column')['cross_split'].agg(['sum', 'count']).reset_index())


,split_column,sum,count
0,split_event_based,0,45
1,split_event_regime_based,0,45
2,split_regime_based,13,45
3,split_substation_based,5,45
4,split_time_based,8,45


## 8. 산출물 저장

Agent와 Priority Engine이 사용할 수 있는 risk score table, metric, threshold, feature importance, 모델 파일을 저장한다.


In [9]:
event_context_output_columns = [
    'days_since_last_fault_event',
    'has_previous_fault_event',
    'recent_fault_30d',
    'recent_fault_60d',
    'recent_fault_90d',
    'days_since_last_task_event',
    'has_previous_task_event',
    'recent_task_30d',
    'recent_task_60d',
    'recent_task_90d',
    'days_since_last_any_event',
    'has_previous_any_event',
    'recent_any_30d',
    'recent_any_60d',
    'recent_any_90d',
    'post_fault_recent_90d',
    'post_task_recent_90d',
]
output_columns = [
    'manufacturer',
    'substation_id',
    'source_file',
    'window_start',
    'window_end',
    'label',
    'fault_label',
    'fault_event_id',
    'configuration_type',
    'estimated_lead_time_hours',
    'lead_time_bucket',
    'anomaly_score',
    'anomaly_threshold',
    'anomaly_label',
    'risk_score',
    'risk_probability',
    'risk_level',
    'main_abnormal_features',
    'related_fault_history',
    'related_disturbance_history',
    'model_explanation_features',
    'maintenance_related',
    'disturbance_count',
    *event_context_output_columns,
    'split_event_based',
    'split_event_regime_based',
    'split_regime_based',
    'split_time_based',
    'split_substation_based',
]
output_columns = [column for column in output_columns if column in modeling_df.columns]
risk_scores_df = modeling_df[output_columns].copy()
risk_scores_df['model_version'] = MODEL_VERSION

run_consistency_df = pd.DataFrame([
    {'table': 'trainable_windows', 'row_count': len(trainable_windows)},
    {'table': 'anomaly_scores', 'row_count': len(anomaly_scores)},
    {'table': 'modeling_df', 'row_count': len(modeling_df)},
    {'table': 'risk_scores_df', 'row_count': len(risk_scores_df)},
])

model_metadata = {
    'model_version': MODEL_VERSION,
    'model_type': 'LightGBM LGBMClassifier',
    'input_feature_path': str(TRAINABLE_WINDOWS_PATH),
    'input_anomaly_score_path': str(ANOMALY_SCORES_PATH),
    'input_fault_alignment_path': str(FAULT_ALIGNMENT_PATH),
    'input_disturbance_alignment_path': str(DISTURBANCE_ALIGNMENT_PATH),
    'risk_target_definition': 'label == pre_fault; fault-report pre-window risk-pattern similarity label',
    'primary_split_column': PRIMARY_SPLIT_COLUMN,
    'event_split_policy': 'pre_fault rows are split by fault_event_id chronology; normal rows use split_regime_based for primary evaluation and split_time_based for legacy comparison',
    'event_context_policy': 'uses only fault/task events before window_start; future event distance is excluded to avoid label leakage',
    'event_context_sentinel_days': EVENT_CONTEXT_SENTINEL_DAYS,
    'recent_event_windows_days': RECENT_EVENT_WINDOWS_DAYS,
    'feature_count': len(model_feature_columns),
    'excluded_feature_columns': excluded_feature_columns,
    'drift_feature_candidates': DRIFT_FEATURE_CANDIDATES,
    'drift_feature_exclusion_enabled': ENABLE_DRIFT_FEATURE_EXCLUSION_EXPERIMENT,
    'drift_excluded_feature_columns': drift_excluded_feature_columns,
    'model_feature_columns': model_feature_columns,
    'context_feature_columns': context_feature_columns,
    'event_context_feature_columns': [column for column in event_context_output_columns if column in model_feature_columns],
    'risk_thresholds': risk_thresholds,
    'threshold_selection_basis': 'fixed operating thresholds from follow-up tuning: medium=0.22, high=0.44, critical=0.90',
    'global_explanation_features': global_explanation_features,
    'run_consistency': run_consistency_df.to_dict(orient='records'),
}

risk_scores_df.to_csv(RISK_SCORES_PATH, index=False)
metrics_df.to_csv(RISK_METRICS_PATH, index=False)
thresholds_df.to_csv(RISK_THRESHOLDS_PATH, index=False)
threshold_selection_df.to_csv(THRESHOLD_SELECTION_PATH, index=False)
feature_importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False)
leakage_audit_df.to_csv(LEAKAGE_AUDIT_PATH, index=False)
run_consistency_df.to_csv(RUN_CONSISTENCY_PATH, index=False)

joblib.dump(risk_model, MODEL_PATH)
MODEL_METADATA_PATH.write_text(json.dumps(model_metadata, ensure_ascii=False, indent=2), encoding='utf-8')

print('saved files:')
print(RISK_SCORES_PATH)
print(RISK_METRICS_PATH)
print(RISK_THRESHOLDS_PATH)
print(THRESHOLD_SELECTION_PATH)
print(FEATURE_IMPORTANCE_PATH)
print(LEAKAGE_AUDIT_PATH)
print(RUN_CONSISTENCY_PATH)
print(MODEL_PATH)
print(MODEL_METADATA_PATH)



saved files:
C:\Project3\HeatGrid_Agent\data\processed\ml_risk\lgbm_risk_scores.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_risk\lgbm_risk_metrics.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_risk\lgbm_risk_thresholds.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_risk\lgbm_threshold_selection.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_risk\lgbm_feature_importance.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_risk\event_split_leakage_audit.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_risk\lgbm_run_consistency.csv
C:\Project3\HeatGrid_Agent\data\processed\ml_risk\models\lightgbm_risk_model.joblib
C:\Project3\HeatGrid_Agent\data\processed\ml_risk\models\risk_model_metadata.json


## 2026-06-25 promotion note

- overall winner: `thermal_group_zscore_only`
- manufacturer 2 / SH winner: `event_context_only`
- hybrid promoted candidate was tested, but overall holdout was worse than current calibrated official chain
- keep official downstream input as `data/processed/ml_risk/lgbm_risk_scores_calibrated.csv` with `risk_level_calibrated`
- promotion review doc: `PREPROCESSING/docs/06_promotion_decision.md`
